## Import Libraries

In [113]:
import pandas as pd
import re
import ipaddress

## Load the dataset

In [115]:
language_data= pd.read_csv("Data.csv")

## Display the dataset

In [117]:
language_data.head(10)

,Data
0,Watch or listen live weekdays at 8:30am MT at ...
1,Watch or listen live weekdays at 8:30am MT at ...
2,"Chubby And Hot, Always Stir The Pot!"
3,"Chubby And Hot, Always Stir The Pot!"
4,"Journalist, publisher of Rebel News — telling ..."
5,Edmonton City Councillor for #WardMétis. #YEG ...
6,Albertan. Mom to children & dogs. Wife. Friend...
7,Albertan. Mom to children & dogs. Wife. Friend...
8,Albertan. Mom to children & dogs. Wife. Friend...
9,Albertan. Mom to children & dogs. Wife. Friend...


In [118]:
language_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7339 entries, 0 to 7338
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Data    7339 non-null   object
dtypes: object(1)
memory usage: 57.5+ KB


## Data Preprocessing

Check for null values

In [121]:
print(language_data.isnull().sum())

Data    0
dtype: int64


Check for duplicate values

In [123]:
duplicate_count = language_data.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

Number of duplicate rows: 5353


In [124]:
language_data = language_data.drop_duplicates()

In [125]:
print("Current shape of dataset:", language_data.shape)

Current shape of dataset: (1986, 1)


In [126]:
duplicate_count = language_data.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

Number of duplicate rows: 0


## Q.No.  A.6

#### What is the total number of ip addresses across all the records?

In [129]:
text_data = ' '.join(language_data['Data'].dropna().astype(str))

# Regex patterns for different IP formats
ipv4_pattern = r'\b(?:(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)\.){3}(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)\b'
cidr_pattern = r'\b(?:(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)\.){3}(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)/[0-9]{1,2}\b'
ipv6_full = r'\b(?:[A-Fa-f0-9]{1,4}:){7}[A-Fa-f0-9]{1,4}\b'
ipv6_compressed = r'\b(?:[A-Fa-f0-9]{1,4}:){1,7}:|::(?:[A-Fa-f0-9]{1,4}:){1,7}[A-Fa-f0-9]{1,4}\b'
ipv6_mapped_ipv4 = r'\b::ffff:(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)\.(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)\.(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)\.(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)\b'

# Additional non-standard IPv4 formats
ipv4_integer = r'\b\d{8,10}\b'                      # 32-bit decimal integer
ipv4_hex = r'\b0x[0-9A-Fa-f]{8}\b'                  # Hexadecimal
ipv4_octal = r'\b0[0-7]+\.[0-7]+\.[0-7]+\.[0-7]+\b' # Octal

# Combine all regexes
all_patterns = f"{ipv4_pattern}|{cidr_pattern}|{ipv6_full}|{ipv6_compressed}|{ipv6_mapped_ipv4}|{ipv4_integer}|{ipv4_hex}|{ipv4_octal}"
matches = re.findall(all_patterns, text_data)

# Normalize and validate
valid_ips = []

for match in matches:
    ip = next(filter(None, match))  # Get non-empty group from match tuple
    try:
        # Handle hex (e.g., 0xC0A80101)
        if ip.startswith("0x"):
            ip = ipaddress.IPv4Address(int(ip, 16)).compressed
        # Handle decimal integer
        elif re.fullmatch(ipv4_integer, ip):
            ip = ipaddress.IPv4Address(int(ip)).compressed
        # Handle octal
        elif re.fullmatch(ipv4_octal, ip):
            parts = [str(int(part, 8)) for part in ip.split('.')]
            ip = '.'.join(parts)
            ipaddress.IPv4Address(ip)  # validate
        else:
            ipaddress.ip_network(ip, strict=False)
        valid_ips.append(ip)
    except ValueError:
        continue


print(f"Total valid IP addresses (all formats): {len(valid_ips)}")

Total valid IP addresses (all formats): 0


#### RESULT- There are no IPV6 or IPV4 or any of the above checked IP address found in the file

## Q.No A.8

#### What is the total number of hashtags across all these tweets?

In [133]:
hashtags = re.findall(r'#\w+', text_data)
print(f"Total count of hashtags across all rows/records: {len(hashtags)}")

Total count of hashtags across all rows/records: 830


#### RESULT- 830 Hashtags were found across the tweets in the dataset